# Flow Past an Obstacle with the Amplitude-Based BGK QLBM

This notebook simulates a uniform flow interacting with a solid obstacle using
`ABBGKQLBM`, the amplitude-based algorithm with a $\tau=1$ angle-encoded BGK collision.
See `ab_bgk_taylor_green.ipynb` for an introduction to the encoding.

The point of interest here is boundary conditions. The collision splits the state into a
physical marker-one sector and an auxiliary marker-zero sector, so both streaming and
reflection are restricted to the physical sector; `ABBGKQLBM` wires that up on its own.
The result is compared against a classical $D_2Q_9$ BGK solver with the same periodic
outer boundary and the same halfway bounce-back rule.

The domain is periodic and unforced, so this is a verification problem rather than a
driven channel-flow benchmark: the uniform stream interacts with the obstacle and
gradually decays.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle
from qiskit_aer import AerSimulator

from qlbm.components import (
    ABBGKQLBM,
    ABBGKInitialConditions,
    ABBGKMeasurement,
    EmptyPrimitive,
)
from qlbm.infra import QiskitRunner, SimulationConfig
from qlbm.infra.reinitialize import ABBGKReinitializer
from qlbm.lattice import ABBGKLattice
from qlbm.tools.utils import create_directory_and_parents

## The lattice and its geometry

The obstacle is declared exactly as for any other amplitude-based lattice. An
`ABBGKLattice` accepts a single geometry: unlike `ABLattice`, it cannot use markers to
label parallel geometries, because its marker is reserved for the physical sector.

In [ ]:
NUM_GRIDPOINTS = 16

OBSTACLE = {
    "shape": "cuboid",
    "x": [7, 8],
    "y": [7, 8],
    "boundary": "bounceback",
}

lattice = ABBGKLattice(
    {
        "lattice": {
            "dim": {"x": NUM_GRIDPOINTS, "y": NUM_GRIDPOINTS},
            "velocities": "d2q9",
        },
        "geometry": [OBSTACLE],
    }
)

output_dir = f"qlbm-output/ab-bgk-obstacle-d2q9-{NUM_GRIDPOINTS}x{NUM_GRIDPOINTS}"
create_directory_and_parents(output_dir)

encoding = lattice.encoding
solid_mask = ABBGKReinitializer.solid_mask_from_geometry(lattice)

print(lattice)
print(f"qubits              = {lattice.circuit.num_qubits}")
print(f"solid gridpoints    = {solid_mask.sum()}")
print(f"velocity bound U    = {encoding.max_velocity}")

## Initial conditions

A uniform stream along the positive $x$ axis. Solid nodes hold no fluid, so their
density and velocity are set to zero.

In [ ]:
INITIAL_SPEED = 0.05

density = np.ones((NUM_GRIDPOINTS, NUM_GRIDPOINTS))
velocity_x = np.full((NUM_GRIDPOINTS, NUM_GRIDPOINTS), INITIAL_SPEED)
velocity_y = np.zeros((NUM_GRIDPOINTS, NUM_GRIDPOINTS))

density[solid_mask] = 0.0
velocity_x[solid_mask] = 0.0

assert INITIAL_SPEED <= encoding.max_velocity

In [ ]:
cfg = SimulationConfig(
    initial_conditions=ABBGKInitialConditions(lattice, density, velocity_x, velocity_y),
    algorithm=ABBGKQLBM(lattice),
    postprocessing=EmptyPrimitive(lattice),
    measurement=ABBGKMeasurement(lattice),
    target_platform="QISKIT",
    compiler_platform="QISKIT",
    optimization_level=0,
    statevector_sampling=True,
    execution_backend=AerSimulator(method="statevector"),
    sampling_backend=AerSimulator(method="statevector"),
)

In [ ]:
cfg.prepare_for_simulation()

In [ ]:
# Number of shots to simulate for each timestep when running the circuit
NUM_SHOTS = 2**14

# Number of timesteps to simulate
NUM_STEPS = 10

In [ ]:
runner = QiskitRunner(cfg, lattice, save_statevector_to_disk=True)

result = runner.run(
    NUM_STEPS,  # Number of time steps
    NUM_SHOTS,  # Number of shots per time step
    output_dir,
    statevector_snapshots=True,
)

## Diagnostics

`solid_population` is the boundary-condition check specific to this demo: exact
reflection leaves the interior of the obstacle empty, so any population found there
would mean particles leaked into the solid.

In [ ]:
for name, value in runner.reinitializer.diagnostics.items():
    print(f"{name:<30} = {value:.3e}")

## Comparison with a classical solver

The reference solver applies the same rule as `ABBounceBackReflectionOperator`: a
population that would stream into a solid node is sent back along the opposite channel.

In [ ]:
def classical_timestep(encoding, populations, solid_mask=None):
    """Advance one classical tau=1 D2Q9 BGK time step.

    At tau=1 the collision is simply a relaxation onto the local equilibrium, which is
    then streamed with periodic outer boundaries. Populations that would stream into a
    solid node are reflected back along the opposite channel, which is the same halfway
    bounce-back rule that ABReflectionOperator implements.
    """
    num_x, num_y, _ = populations.shape
    velocities = encoding.velocities.astype(int)
    opposite = [0, 3, 4, 1, 2, 7, 8, 5, 6]

    density, velocity_x, velocity_y = encoding.macroscopic(populations, solid_mask)
    collided = encoding.equilibrium(density, velocity_x, velocity_y)

    if solid_mask is not None:
        collided[solid_mask, :] = 0.0

    streamed = np.zeros_like(collided)

    for x in range(num_x):
        for y in range(num_y):
            if solid_mask is not None and solid_mask[x, y]:
                continue

            for velocity, (shift_x, shift_y) in enumerate(velocities):
                target_x, target_y = (x + shift_x) % num_x, (y + shift_y) % num_y

                if solid_mask is not None and solid_mask[target_x, target_y]:
                    streamed[x, y, opposite[velocity]] += collided[x, y, velocity]
                else:
                    streamed[target_x, target_y, velocity] += collided[x, y, velocity]

    return streamed

In [ ]:
populations = encoding.equilibrium(density, velocity_x, velocity_y)
populations[solid_mask, :] = 0.0

for _ in range(NUM_STEPS):
    populations = classical_timestep(encoding, populations, solid_mask)

classical = encoding.macroscopic(populations, solid_mask)
quantum = (result.density, result.velocity_x, result.velocity_y)

fluid = ~solid_mask
velocity_error = np.linalg.norm(
    np.stack(
        [
            quantum[1][fluid] - classical[1][fluid],
            quantum[2][fluid] - classical[2][fluid],
        ]
    )
) / np.linalg.norm(np.stack([classical[1][fluid], classical[2][fluid]]))

print(f"time steps                 = {NUM_STEPS}")
print(f"relative L2 velocity error = {velocity_error:.3e}")
print(f"maximum solid population   = {np.max(np.abs(quantum[0][solid_mask])):.3e}")

In [ ]:
def compare_flow_fields(quantum, classical, solid_mask, obstacle):
    """Plot the quantum and classical speed fields side by side with their difference."""
    quantum_speed = np.sqrt(quantum[1] ** 2 + quantum[2] ** 2)
    classical_speed = np.sqrt(classical[1] ** 2 + classical[2] ** 2)
    error = np.sqrt((quantum[1] - classical[1]) ** 2 + (quantum[2] - classical[2]) ** 2)

    figure, axes = plt.subplots(1, 3, figsize=(15, 4.4), constrained_layout=True)
    panels = (
        ("ABBGKQLBM", quantum_speed, "viridis"),
        ("Classical D2Q9 BGK", classical_speed, "viridis"),
        ("|u_quantum - u_classical|", error, "magma"),
    )
    speed_limit = max(quantum_speed.max(), classical_speed.max())

    for axis, (title, field, colormap) in zip(axes, panels):
        image = axis.imshow(
            field.T,
            origin="lower",
            cmap=colormap,
            vmin=0.0,
            vmax=speed_limit if colormap == "viridis" else None,
        )
        axis.set_title(title)
        axis.set_xlabel("x")
        axis.set_ylabel("y")
        figure.colorbar(image, ax=axis, shrink=0.85)

        axis.add_patch(
            Rectangle(
                (obstacle["x"][0] - 0.5, obstacle["y"][0] - 0.5),
                obstacle["x"][1] - obstacle["x"][0] + 1,
                obstacle["y"][1] - obstacle["y"][0] + 1,
                facecolor="0.35",
                edgecolor="white",
                linewidth=1.0,
            )
        )

    return figure

In [ ]:
compare_flow_fields(quantum, classical, solid_mask, OBSTACLE);

## Output

The geometry is exported as an `stl` file alongside the `step_<x>.vti` time step files,
so the obstacle can be rendered together with the flow field in ParaView.

In [ ]:
print(f"ParaView output = {output_dir}/paraview")